# 🔋Predicting Battery Degradation Through Cycling Data🔋

## Contributions
This is a solo project done by Lixin Nie


## Introduction

Lithium-ion batteries are widely used in modern applications, including consumer electronics, electric vehicles, and energy storage systems. A key challenge in their usage is performance degradation over time, which affects capacity, efficiency, and overall lifespan.

In this project, we analyze battery cycling data to better understand charge and discharge behavior and prepare the data for future degradation modeling. The dataset used comes from experimental battery tests and includes measurements such as voltage, current, temperature, and cycle-related information over time.

The goal of this notebook is to:
- Load and inspect the raw battery dataset
- Clean and preprocess the data for analysis
- Identify key features such as charge and discharge phases
- Prepare the dataset for further analysis, including cycle extraction and degradation modeling

By structuring and cleaning the data properly, we enable more meaningful analysis and modeling in later stages of the analysis.

# Data Collection

The data collection stage of the Data Science lifecycle involves gathering relevant data that will help answer our research questions. For this project, we used battery cycling data provided by the **Center for Advanced Life Cycle Engineering (CALCE)** at the University of Maryland, available at [https://calce.umd.edu/battery-data](https://calce.umd.edu/battery-data). CALCE provides open-access experimental data on lithium-ion battery performance, including continuous full and partial cycling, storage conditions, dynamic driving profiles, open-circuit voltage measurements, and impedance data. Battery form factors in the dataset include cylindrical, pouch, and prismatic cells.

The dataset includes high-resolution measurements such as voltage, current, capacity, temperature, and cycle count. These variables allow us to track how battery performance evolves over time and under repeated usage, which is critical for analyzing degradation patterns.

Since the data is collected and shared by a reputable research institution, it is considered reliable and suitable for scientific analysis. For this project, we selected a subset of the available cycling datasets that contain sufficient observations and relevant features for studying battery degradation. These datasets were then downloaded and prepared for further processing in the next stage of the data science lifecycle.

We are going to utulize Python modules such as **numpy**, **pandas**, and **matplotlib** were used to process, analyze, and visualize the data.

In [70]:
#importing the necessary packages
import numpy as np 
import pandas as pd
import matplotlib as plt


The dataset is stored in a `.txt` file format. Unlike standard CSV files, this dataset uses a tab (`\t`) as the delimiter. Therefore, we must explicitly specify the separator when loading the data using pandas.

We load the dataset into a pandas DataFrame for further processing and analysis.

In [71]:
#We would need additional argument "sep="\t"" since the seperator is not comma in this case. 
df = pd.read_csv("./CS2_8/CS2_8_1_19_10.txt", sep="\t")

#------------- Generate Informations ------------------#

print("Shape of dataframe (rows, columns):")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDataframe info:")
print(df.info())
print("\nPreview of dataframe:")
df.head()

Shape of dataframe (rows, columns):
(3268, 30)

Column names:
['Time', 'Status code', 'Status category', 'Status color', 'Pgm code', 'Pgm step', 'Pgm para', 'Pgm cycle', 'mV', 'mA', 'Temperature', 'Duration', 'Charge count', 'Discharge count', 'Capacity', 'Analog input 1', 'Analog input 2', 'Analog input 3', 'Analog input 4', 'Digital input 1', 'Digital input 2', 'Digital input 3', 'Digital input 4', 'Digital output 1', 'Digital output 2', 'Digital output 3', 'Digital output 4', 'Analog output 1', 'Analog output 2', 'Unnamed: 29']

Dataframe info:
<class 'pandas.DataFrame'>
RangeIndex: 3268 entries, 0 to 3267
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Time              3268 non-null   float64
 1   Status code       3268 non-null   int64  
 2   Status category   3268 non-null   int64  
 3   Status color      3268 non-null   int64  
 4   Pgm code          3268 non-null   int64  
 5   Pgm step        

,Time,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,mV,mA,...,Digital input 2,Digital input 3,Digital input 4,Digital output 1,Digital output 2,Digital output 3,Digital output 4,Analog output 1,Analog output 2,Unnamed: 29
0,0.000000,8,3,3,0,1,2,2,4005,552,...,0,0,0,0,0,0,0,0,0,NaN
1,0.750233,8,3,3,0,1,2,2,4030,550,...,0,0,0,0,0,0,0,0,0,NaN
2,1.745283,8,3,3,0,1,2,2,4042,548,...,0,0,0,0,0,0,0,0,0,NaN
3,2.761917,8,3,3,0,1,2,2,4047,549,...,0,0,0,0,0,0,0,0,0,NaN
4,3.758400,8,3,3,0,1,2,2,4051,550,...,0,0,0,0,0,0,0,0,0,NaN


Proper data cleaning is a critical step in the data science pipeline. Even when datasets appear structured, unnecessary columns, inconsistent naming, and unclear units can hinder analysis and lead to errors.

For this specific dataframe, we are going to drop then columns are not useful for our analysis.

In [72]:
#Dropping the useless null columns
df = df.drop(columns=['Unnamed: 29'])

#Theres quite a bit of the all zero columns, we are going to use loops to identify them.
io_cols = [c for c in df.columns if any(
    c.startswith(prefix) for prefix in
    ['Analog input', 'Analog output', 'Digital input', 'Digital output']
)]

zero_io_cols = [c for c in io_cols if (df[c] == 0).all()]
df = df.drop(columns=zero_io_cols)

#Inspect cleaned df 
print("Shape of dataframe (rows, columns):")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nPreview of dataframe:")

df.head()


Shape of dataframe (rows, columns):
(3268, 15)

Column names:
['Time', 'Status code', 'Status category', 'Status color', 'Pgm code', 'Pgm step', 'Pgm para', 'Pgm cycle', 'mV', 'mA', 'Temperature', 'Duration', 'Charge count', 'Discharge count', 'Capacity']

Preview of dataframe:


,Time,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,mV,mA,Temperature,Duration,Charge count,Discharge count,Capacity
0,0.000000,8,3,3,0,1,2,2,4005,552,19,2,1,0,0
1,0.750233,8,3,3,0,1,2,2,4030,550,20,47,1,0,0
2,1.745283,8,3,3,0,1,2,2,4042,548,19,107,1,0,0
3,2.761917,8,3,3,0,1,2,2,4047,549,20,168,1,0,0
4,3.758400,8,3,3,0,1,2,2,4051,550,20,228,1,0,0


Convert mV→V and mA→A to match standard SI units; otherwise calculations like capacity and energy will be off by 1000× and incompatible with formulas, plots, and libraries. Also standardize time units: here, Time is in minutes and Duration in seconds. Converting both to seconds avoids errors in cycle analysis and enables correct computation of metrics like C-rate and time-based features.

In [73]:
#Convertin time to seconds
df['Time'] = df['Time'] * 60

df = df.rename(columns={
    'Time':     'Time_s',
    'Duration': 'Duration_s'
})

#Scale the electrical units
df['mV'] = df['mV'] / 1000
df['mA'] = df['mA'] / 1000

df = df.rename(columns={
    'mV': 'Voltage_V',
    'mA': 'Current_A'
})

df.head()

,Time_s,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,Voltage_V,Current_A,Temperature,Duration_s,Charge count,Discharge count,Capacity
0,0.00000,8,3,3,0,1,2,2,4.005,0.552,19,2,1,0,0
1,45.01398,8,3,3,0,1,2,2,4.030,0.550,20,47,1,0,0
2,104.71698,8,3,3,0,1,2,2,4.042,0.548,19,107,1,0,0
3,165.71502,8,3,3,0,1,2,2,4.047,0.549,20,168,1,0,0
4,225.50400,8,3,3,0,1,2,2,4.051,0.550,20,228,1,0,0


Decode phase from the sign of Current_A instead of opaque status codes: positive = charging, negative = discharging. This allows us to tell if the battery is charging to discharging right away, and makes analysis and plots immediately interpretable.

In [74]:
#We are going to instect the unique values for status category first.
print("Status category unique values:", df['Status category'].unique())
df['_current_sign'] = df['Current_A'].apply(
    lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'zero')
)
print(pd.crosstab(df['Status category'], df['_current_sign']))
df = df.drop(columns=['_current_sign'])

Status category unique values: [ 3  6 15  4  0]
_current_sign    negative  positive  zero
Status category                          
0                       0         0     1
3                       0      1903     0
4                       0         1     0
6                    1352         0     0
15                      0        10     1


From the data, we can infer that status 3 corresponds to charging and status 6 to discharging. However, codes such as 0, 4, and 15 are ambiguous and do not clearly indicate the battery’s state. Instead of relying on these unclear status codes, we create a new charge-state column based on the current, using its sign to determine whether the battery is charging or discharging.

In [75]:
# We derive phase from the sign
def label_phase(current):
    if current > 0:
        return 'charge'
    elif current < 0:
        return 'discharge'
    else:
        return 'rest'

df['phase'] = df['Current_A'].apply(label_phase)

# Display result
print(df['phase'].value_counts())
display(df.head())

phase
charge       1914
discharge    1352
rest            2
Name: count, dtype: int64


,Time_s,Status code,Status category,Status color,Pgm code,Pgm step,Pgm para,Pgm cycle,Voltage_V,Current_A,Temperature,Duration_s,Charge count,Discharge count,Capacity,phase
0,0.00000,8,3,3,0,1,2,2,4.005,0.552,19,2,1,0,0,charge
1,45.01398,8,3,3,0,1,2,2,4.030,0.550,20,47,1,0,0,charge
2,104.71698,8,3,3,0,1,2,2,4.042,0.548,19,107,1,0,0,charge
3,165.71502,8,3,3,0,1,2,2,4.047,0.549,20,168,1,0,0,charge
4,225.50400,8,3,3,0,1,2,2,4.051,0.550,20,228,1,0,0,charge


When a battery cycles through charge and discharge, the Capacity column accumulates continuously within each phase. It’s not a single snapshot but a running counter that grows row by row. To determine the total capacity delivered in a given discharge cycle, we need to take the maximum value reached before the counter resets, which represents the full capacity for that cycle. This allows us to condense the time-series data into a single meaningful value per cycle, making it easier to track performance over time and analyze battery degradation. To do this, we are going to build a summary table that illustrates the degragation of max battery capacity. 

In [76]:
#Let's first inspect Capacity and Pgm cycle
print("Pgm cycle unique values:", sorted(df['Pgm cycle'].unique()))
print()

#Look at how Capacity behaves within a single cycle
sample_cycle = df[df['Pgm cycle'] == 1]
print(f"Cycle 1 row count: {len(sample_cycle)}")
print(sample_cycle[['Time_s', 'Pgm cycle', 'phase', 'Current_A', 'Capacity']].head())
print("...")
print(sample_cycle[['Time_s', 'Pgm cycle', 'phase', 'Current_A', 'Capacity']].tail())

Pgm cycle unique values: [np.int64(0), np.int64(1), np.int64(2)]

Cycle 1 row count: 1352
         Time_s  Pgm cycle      phase  Current_A  Capacity
149  8904.12498          1  discharge     -0.287         0
150  8926.06998          1  discharge     -0.549         0
151  8986.54998          1  discharge     -0.550         1
152  9045.56202          1  discharge     -0.549         1
153  9105.95502          1  discharge     -0.549         2
...
            Time_s  Pgm cycle      phase  Current_A  Capacity
3261  193739.64498          1  discharge     -0.550        96
3262  193800.03798          1  discharge     -0.549        97
3263  193860.34602          1  discharge     -0.551        98
3264  193920.82602          1  discharge     -0.549        98
3265  193979.83698          1  discharge     -0.551        99


It seems like "Pgm cycle" only has 3 values (0, 1, 2). So it's not tracking individual cycles. Instead, cycle counter is accounted by the "Discharge count" column. Notice that Capacity resets to 0  multiple times within "Pgm cycle" = 1.  So we are going to group by "Discharge count" Instead.

In [77]:
#Let's Verify Discharge count is the right grouping variable
print("Discharge count unique values:", sorted(df['Discharge count'].unique()))
print("Max discharge count:", df['Discharge count'].max())

print(df[df['phase'] == 'discharge'][['Discharge count', 'Capacity']].groupby('Discharge count').agg(['min','max','count']))

Discharge count unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
Max discharge count: 11
                Capacity           
                     min  max count
Discharge count                    
1                      0  100   124
2                      0  101   124
3                      0  100   123
4                      0  100   123
5                      0  100   123
6                      0  100   123
7                      0  100   123
8                      0   99   122
9                      0   99   123
10                     0   99   122
11                     0   99   122


Looks like Discharge count are consistent with showing battery degragtation, we are going to proceed with building a summary table that describes this relationship. 

In [78]:
df_discharge = df[df['phase'] == 'discharge']

df_cycles = (
    df_discharge
    .groupby('Discharge count')['Capacity']
    .max()
    .reset_index()
    .rename(columns={
        'Discharge count': 'cycle_number',
        'Capacity':        'peak_capacity'
    })
)

# Dropping cycle 0, no discharge data
df_cycles = df_cycles[df_cycles['cycle_number'] > 0].reset_index(drop=True)

print(df_cycles.to_string())

    cycle_number  peak_capacity
0              1            100
1              2            101
2              3            100
3              4            100
4              5            100
5              6            100
6              7            100
7              8             99
8              9             99
9             10             99
10            11             99
